# 04 - Clustering Algorithms and Choosing K

Compare six clustering methods and evaluate them with internal quality metrics.


## Definition
Explain the concept in one sentence and why it matters in delivery analytics.

## Theory
Describe the assumptions and algorithmic behavior at a practical level.

## Mathematical Intuition
Show the formula or geometry intuition behind the method.

## Visual Explanation
Use a chart/map to make the concept concrete.

## Code Explanation
Walk through what each code block does and what inputs/outputs mean.

## Interpretation of Results
Connect metrics and visuals to operational decisions.


In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

if (Path.cwd() / "src").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd().parent / "src").exists():
    PROJECT_ROOT = Path.cwd().parent
else:
    raise RuntimeError("Could not locate project root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


Project root: /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/Core Machine Learning and Data Science/Geospatial Clustering


In [2]:
from src.config import TRAIN_FILE_PATH
from src.data_loader import load_raw_data, load_and_clean_data, explain_dataset_fields


In [3]:
from src.features import build_clustering_features, select_features
from src.outlier_detection import detect_outliers
from src.clustering import run_all
from src.evaluation import evaluate_all, k_selection_diagnostics
from src.visualization import plot_k_selection_metrics, plot_elbow_curve

df = load_and_clean_data(TRAIN_FILE_PATH)
feat = build_clustering_features(df.copy())
X = select_features(feat)
geo = feat[["Restaurant_latitude", "Restaurant_longitude"]].to_numpy(dtype=float)

consensus, _ = detect_outliers(feat, feature_matrix=X)
keep = ~consensus.mask

feat = feat.loc[keep].reset_index(drop=True)
X = X[keep]
geo = geo[keep]

results = run_all(X, geo_coords=geo)
eval_df = evaluate_all(X, results, geo_coords=geo)
display(eval_df)

diag_df = k_selection_diagnostics(X)
display(diag_df.head())
print(plot_k_selection_metrics(diag_df, filename="nb_k_selection_metrics.png"))
elbow_df = diag_df.rename(columns={"k":"n_clusters"})[["n_clusters","inertia","silhouette"]]
print(plot_elbow_curve(elbow_df, filename="nb_elbow.png"))


,algorithm,n_clusters,n_noise,noise_ratio,silhouette,davies_bouldin,calinski_harabasz,stability_ari,composite_score
0,hdbscan,241.0,248.0,0.006441,0.081708,20.760860,258362.645022,1.000000,3.782774
1,agglomerative,6.0,0.0,0.000000,0.054710,6.029575,5264.967221,1.000000,3.093476
2,gmm,6.0,0.0,0.000000,0.110457,3.876809,5644.931764,0.751795,2.985512
3,kmeans,6.0,0.0,0.000000,0.161624,130.426519,5581.080484,0.754665,2.417783
4,minibatch_kmeans,6.0,0.0,0.000000,0.083363,5.041722,5668.424530,0.579521,2.341272
5,dbscan,220.0,592.0,0.015375,0.041430,11.337326,34887.649421,1.000000,2.058087


,k,inertia,silhouette,davies_bouldin,calinski_harabasz
0,2.0,95295.966946,-0.003311,126.072753,1.537158
1,3.0,85704.952442,0.091199,3.339726,4870.721236
2,4.0,79414.228756,0.108484,3.093290,5139.390938
3,5.0,76433.835905,0.045514,9.225715,4954.096570
4,6.0,70006.376082,0.161624,130.426519,5581.080484


/home/ahmad/AI/Github/40 AI-ML Projects for Beginners/Core Machine Learning and Data Science/Geospatial Clustering/outputs/plots/nb_k_selection_metrics.png
/home/ahmad/AI/Github/40 AI-ML Projects for Beginners/Core Machine Learning and Data Science/Geospatial Clustering/outputs/plots/nb_elbow.png


**Interpretation**
- Do not pick models from one metric only.
- Evaluate separation, compactness, stability, and noise behavior together.
